In [1]:
try:
    kjghfdhdsfh3
    assert False
except NameError:
    kjghfdhdsfh3 = 3
    pass
import os

In [ ]:
list(range(6,24))

In [ ]:
kjghfdhdsfh3 = 3
import os, importlib
import sys, pathlib
import psutil
import threadpoolctl
# Get the current process
import numba
# numba.config.THREADING_LAYER = 'forksafe'

sys.path += ["../diff_co_mpc/"]
workstation = True
# sys.path += [str(pathlib.Path(os.getcwd()) / '..' / "diff_co_mpc")]
if workstation:
    # LD_PRELOAD=<path>/libgomp.so
    # os.environ['LD_PRELOAD'] = '/usr/lib/x86_64-linux-gnu/libmkl_core.so:/usr/lib/x86_64-linux-gnu/libmkl_intel_thread.so:/usr/lib/x86_64-linux-gnu/libmkl_intel_lp64.so:/usr/lib/x86_64-linux-gnu/libiomp5.so:'
    os.environ["OMP_NUM_THREADS"] = "12"  #
    os.environ["MKL_NUM_THREADS"] = "12"  #    
    # os.environ['TORCH_LOGS'] = '+dynamic'
    os.environ["OMP_PLACES"] = "{12:23}"
    os.environ["OMP_PROC_BIND"] = "true"
    os.environ['MKL_THREADING_LAYER'] = 'GNU'
    p = psutil.Process()
    p.cpu_affinity(list(range(12,24)))
else:
    p = psutil.Process()
    # LD_PRELOAD=<path>/libgomp.so
    os.environ["OMP_NUM_THREADS"] = "3"  #
    os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
    os.environ["TORCHINDUCTOR_COMPILE_THREAD"] = "1"
    # TORCH_LOGS=dynamic
    os.environ['TORCH_LOGS'] = '+dynamic'
    os.environ["OMP_PLACES"] = "{0,2,4}"
    os.environ["OMP_PROC_BIND"] = "true"
    p.cpu_affinity([0,1,2,3,4,5])
import casadi as ca
from IPython.display import clear_output, display, SVG
from functools import partial
from toolz import (
    memoize,
    pipe,
    accumulate,
    groupby,
    compose,
    compose_left,
    merge,
    first,
)
from toolz.curried import do
from collections import namedtuple
import typing as T
import numpy as np
import torch
import itertools
import pathlib, os
from matplotlib import pyplot as plt
import time
import copy
import threadpoolctl

import importlib
from pydrake.all import (
    ModelInstanceIndex,
    RigidTransform,
    RotationMatrix,
    RollPitchYaw,
    StartMeshcat,
)
import threadpoolctl
from utils.my_casadi.misc import time_function, Jit, Compile


from utils.math.BSpline import BSpline

bspline = BSpline(np.array([[0, 0], [0.5, 0.5], [1, 1]]), 3)
bspline.fast_batch_evaluate(np.linspace(0,1,10))
bspline.fast_create_derivative_spline()

import utils.my_casadi.misc as ca_utils
from utils.my_casadi.misc import veccat

# from diff_co_mpc.casadi_custom import *
# from diff_co_mpc.global_planner import *
from mpc.planner_implementation.make import *
from mpc.optimisation import load_opt_info, DiffCoOptions, make_system
from misc.helper_functions import *

try:
    meshcat
    print("meshcat already defined", meshcat.web_url())
except NameError:
    meshcat = StartMeshcat()
try:
    ROS_INITIALIZED 
except:
    import sys
    sys.path += ["/opt/ros/noetic/lib/python3/dist-packages"]
    import rospy as ros
    ros.init_node('global_planner')
    ROS_INITIALIZED = True
    from actionlib import SimpleActionClient
    from sensor_msgs.msg import JointState
    from trajectory_msgs.msg import JointTrajectory, JointTrajectoryPoint
    from control_msgs.msg import FollowJointTrajectoryAction, \
                                FollowJointTrajectoryGoal, FollowJointTrajectoryResult
TEMP_FOLDER = pathlib.Path(os.getcwd()) / '..' / "temp"
TEMP_FOLDER.mkdir(exist_ok=True)
CODEGEN_FOLDER = TEMP_FOLDER / "codegen"
CODEGEN_FOLDER.mkdir(exist_ok=True, parents=True)
yaml_path = pathlib.Path('..') / 'config'/ "plant_definition.yaml"

system = make_system(yaml_path, meshcat, CODEGEN_FOLDER)
complete_plant = system["complete_plant"]
robots = system["robots"]
viz_helper = system["visualization_helper"]
carried_object = system["carried_object"]
yaml = system["yaml"]

# EE_transform_1 = RigidTransform(
#     p=[0.1, 0, 0], rpy=RollPitchYaw([0, 0, np.pi])
# ).GetAsMatrix4()
# EE_transform_2 = RigidTransform(
#     p=[-0.0, 0, 0], rpy=RollPitchYaw([0, 0, 0])
# ).GetAsMatrix4()


def configuration_to_control_points(configuration, number_of_control_points):
    return np.tile(configuration.reshape(-1,1),(1,number_of_control_points))
def publish_trajectory(opt_data, lcm_subscription_handler, order = 3):
    message = lcmt_global_solve()
    message.utime = time.perf_counter_ns()
    message.bspline_robot_1.control_points = opt_data.result["robot_1_control_points"]
    message.bspline_robot_1.order = order
    message.bspline_robot_2.order = order
    message.bspline_robot_1.num_positions = opt_data.result["robot_1_control_points"].shape[0]
    message.bspline_robot_1.num_control_points = opt_data.result["robot_1_control_points"].shape[1]
    message.bspline_robot_2.control_points = opt_data.result["robot_2_control_points"]
    message.bspline_robot_2.num_positions = opt_data.result["robot_2_control_points"].shape[0]
    message.bspline_robot_2.num_control_points = opt_data.result["robot_2_control_points"].shape[1]
    message.bspline_object.control_points = opt_data.result["carried_object_control_points"]
    message.bspline_object.num_positions = opt_data.result["carried_object_control_points"].shape[0]
    message.bspline_object.num_control_points = opt_data.result["carried_object_control_points"].shape[1]
    lcm_subscription_handler.lcm.Publish("global_trajectory", message.encode())
def display_trajectory(opt_data,meshcat):
    bspline_1_col = BSpline(
        opt_data.result["robot_1_control_points"],
        opt_collision.order,
    )
    bspline_2_col = BSpline(
        opt_data.result["robot_2_control_points"],
        opt_collision.order,
    )
    bspline_obj_col = BSpline(
        opt_data.result["carried_object_control_points"],
        opt_collision.order,
    )
    

    meshcat.StartRecording(frames_per_second=30.0)

    for s in np.linspace(0,1,15):
        q_1 = bspline_1_col.evaluate(s)
        q_1 = np.concatenate([q_1,np.zeros(2)])
        q_2 = bspline_2_col.evaluate(s)
        q_2 = np.concatenate([q_2,np.zeros(2)])
        q_obj = bspline_obj_col.evaluate(s)
        
        viz_helper.set_position('robot_0',q_1)
        viz_helper.set_position('robot_1',q_2)
        viz_helper.set_position('carried_object',q_obj)
        

        viz_helper.diagram_context.SetTime(s)
        
        viz_helper.publish_diagram()

    meshcat.StopRecording()
    
    meshcat.PublishRecording()
def display_initial_guesses(parallel_data_object,show_all = False):
    meshcat.StartRecording(frames_per_second=30.0)
    ii = 0
    bspline_1_col = []
    bspline_2_col = []
    bspline_obj_col = []
    for i in range(parallel_data_object.map_size):
        bspline_1_col.append(BSpline(parallel_data_object.result['robot_1_control_points'][i],opt_collision.order))
        bspline_2_col.append(BSpline(parallel_data_object.result['robot_2_control_points'][i],opt_collision.order))
        bspline_obj_col.append(BSpline(parallel_data_object.result['carried_object_control_points'][i],opt_collision.order))
    for i in range(parallel_data_object.map_size):
        if parallel_data_object.within_bounds()[i] or show_all:
            for s in np.linspace(0,1,80):
                q_1 = bspline_1_col[i].evaluate(s)
                q_2 = bspline_2_col[i].evaluate(s)
                q_obj = bspline_obj_col[i].evaluate(s)
                q_1 = np.concatenate([q_1,np.zeros(2)])
                q_2 = np.concatenate([q_2,np.zeros(2)])
                
                
                viz_helper.set_position('robot_0',q_1)
                viz_helper.set_position('robot_1',q_2)
                viz_helper.set_position('carried_object',q_obj)              

                viz_helper.diagram_context.SetTime(s+ii)
                viz_helper.publish()
            ii+=1
            if parallel_data_object.within_bounds()[i]: print(ii)
    meshcat.StopRecording()
    meshcat.PublishRecording()
def collision_score_from_parallel_solves(num_samples,planner):
    def polyharmonic_kernel(
        x: torch.Tensor, y: torch.Tensor, alpha: int
    ) -> torch.Tensor:

        if alpha % 2 == 1:
            return torch.linalg.norm(x - y, axis=-1) ** alpha
        else:
            r = torch.linalg.norm(x - y, axis=-1)
            temp = (r**alpha) * torch.log(r)
            temp[torch.isnan(temp)] = 0.0

            return temp
    total_scores = {}
    for i in range(planner.parallel_solve_data.map_size):
        bspline_1 = (BSpline(planner.parallel_solve_data.result['robot_1_control_points'][i],opt_collision.order))
        bspline_2 = (BSpline(planner.parallel_solve_data.result['robot_2_control_points'][i],opt_collision.order))
        # bspline_obj_col = (BSpline(planner.parallel_solve_data.result['carried_object_control_points'][i],opt_collision.order))
        total_score = 0
        for s in np.linspace(0,1,num_samples):
            # print(s)
            # text= ''
            for robot in ['robot_1','robot_2']:
                for group_name in ['group_1','group_2','group_3','group_4']:
                    weights = torch.as_tensor(planner.lcm_subscription_handler.last_svm[robot][group_name][-1]['weights'])
                    support_vectors = torch.as_tensor(planner.lcm_subscription_handler.last_svm[robot][group_name][-1]['sv'])
                    polynomial_weights = torch.as_tensor(planner.lcm_subscription_handler.last_svm[robot][group_name][-1]['pol_weights']).squeeze()
                    q = torch.as_tensor(bspline_2.evaluate(s) if robot == 'robot_2' else bspline_1.evaluate(s))
                    col_model = planner.robot_1_collision_model if robot == 'robot_1' else planner.robot_2_collision_model
                    fk_q = col_model.forward_kinematics_groups_torch[group_name](q)
                    sample = fk_q.reshape(-1)
                    dist = polyharmonic_kernel(sample, support_vectors.T, 1)
                    score = torch.dot(weights.reshape(-1).to(torch.float64), dist.reshape(-1).to(torch.float64)) + polynomial_weights[0].reshape(-1).to(torch.float64) + torch.dot(polynomial_weights[1:].reshape(-1).to(torch.float64), sample.reshape(-1).to(torch.float64))
                    total_score += torch.clip(score,torch.tensor(0.),torch.inf)
                    # print(s,score)
                    # text += f'{group_name}: {score.item():.2f} '
                # text+= '\n'
            # print(text)
        total_scores[i] = total_score
    return total_scores
from pydrake.all import Sphere

# path = 'tensors.pth'
# loaded_tensors = torch.load(path)
# # Access the tensors
# obstacle_points = loaded_tensors['tensor1'].numpy()
# obstacle_radii = loaded_tensors['tensor2'].numpy()

# sphere_positions = obstacle_points
from multiprocessing import shared_memory
dtype = np.float32
array_shape = (10000, 3)
shared_memory_name = "point_cloud_shared_memory"

try:
    shm = shared_memory.SharedMemory(name=shared_memory_name, create=True, size=np.prod(array_shape) * np.dtype(dtype).itemsize)
    shared_array = np.ndarray(array_shape, dtype=dtype, buffer=shm.buf)
    shared_array[:] = np.nan
    print("Created new shared memory.")
except FileExistsError:
    shm = shared_memory.SharedMemory(name=shared_memory_name)
    shared_array = np.ndarray(array_shape, dtype=dtype, buffer=shm.buf)
    print("Linked to existing shared memory.")
point_cloud = shared_array.copy()
point_cloud = point_cloud[~np.isnan(point_cloud).any(axis=1)]

point_cloud_size = point_cloud.shape[ 0 ]
obstacle_points = torch.as_tensor(point_cloud).to(device = 'cpu',dtype=torch.float32)
obstacle_radii = torch.tensor(0.04, dtype=torch.float32, device='cpu').expand(point_cloud_size, 1).to(device = 'cpu',dtype=torch.float32)
for i, q_sample in enumerate(obstacle_points):
    meshcat.SetObject (
        f"/obstacles/{i}", Sphere(obstacle_radii[i])
    )
    meshcat.SetProperty (
        f"/obstacles/{i}", "position", q_sample.tolist()
    )
class PointCloudViz:
    def __init__(self, meshcat):
        self.meshcat = meshcat
        self.shm = shared_memory.SharedMemory(name=shared_memory_name)
        self.point_cloud = np.ndarray(array_shape, dtype=dtype, buffer=self.shm.buf)

    def update(self):
        point_cloud = self.point_cloud.copy()
        point_cloud = point_cloud[~np.isnan(point_cloud).any(axis=1)]
        obstacle_radii = torch.tensor(0.04, dtype=torch.float32, device='cpu').expand(point_cloud.shape[0], 1).to(device = 'cpu',dtype=torch.float32).numpy()
        for i in range(1000):
            self.meshcat.Delete(f"/obstacles/{i}")
        for i, q_sample in enumerate(point_cloud):
            self.meshcat.SetObject(
                f"/obstacles/{i}", Sphere(obstacle_radii[i])
            )
            self.meshcat.SetProperty(
                f"/obstacles/{i}", "position", q_sample.tolist()
            )
point_cloud_viz = PointCloudViz(meshcat)
point_cloud_viz.update()



In [2]:
collision_options = {
    "print_time": True,
    "ipopt": {
        "print_level": 3,
        "hessian_approximation": "exact",
        "linear_solver": "ma57",
        "max_iter": 10000,
        # "print_timing_statistics":"yes",
        "constr_viol_tol": 1e-4,
        "tol": 1e-4,
        "acceptable_tol": 1e-2,
        "acceptable_obj_change_tol": 1e-1,
        "max_wall_time": 10.,
        "warm_start_init_point": "yes",
        # 'warm_start_same_structure':'no',
        'warm_start_bound_push'  :     1e-9,
        'warm_start_bound_frac':       1e-9,
        'warm_start_slack_bound_frac': 1e-9,
        'warm_start_slack_bound_push': 1e-9,
        'warm_start_mult_bound_push' : 1e-9,
        'nlp_scaling_method':'none',
        'mu_strategy':'monotone',
        'mu_init':0.001,
        # 'mu_init':0.1,
        'ma57_pre_alloc':3.
        # "fast_step_computation": "yes",
    },
}
collision_options_very_short = {
    "ipopt": {
        "print_level": 0,
        "hessian_approximation": "exact",
        "linear_solver": "ma57",
        "max_iter": 10000,
        # "print_timing_statistics":"yes",
        "constr_viol_tol": 1e-4,
        "tol": 1e-4,
        "acceptable_tol": 1e-2,
        "acceptable_obj_change_tol": 1e-1,
        "max_wall_time": 0.25,
        "warm_start_init_point": "yes",
        # 'warm_start_same_structure':'no',
        'warm_start_bound_push'  :     1e-9,
        'warm_start_bound_frac':       1e-9,
        'warm_start_slack_bound_frac': 1e-9,
        'warm_start_slack_bound_push': 1e-9,
        'warm_start_mult_bound_push' : 1e-9,
        'nlp_scaling_method':'none',
        'mu_strategy':'monotone',
        'mu_init':0.001,
        # 'mu_init':0.1,
        'ma57_pre_alloc':3.
        # "fast_step_computation": "yes",
    },
}

num_samples = 10
num_control_points = 20
order = 3

import importlib
import mpc
import mpc.planner_implementation.make

importlib.reload(mpc.optimisation)
importlib.reload(mpc.planner_implementation.make)
from mpc.optimisation import *

# from diff_co_mpc.mpc import make_planner_with_collision, make_planner_with_no_collision
from mpc.planner_implementation.make import *

diff_co_options = DiffCoOptions.from_yaml(pathlib.Path(yaml_path))

gaze_options = VisionOptions.from_yaml(pathlib.Path(yaml_path))

opt_with_collision_compile_path = CODEGEN_FOLDER / "nlp_with_collision_vision"
opt_with_collision_compile_path.mkdir(exist_ok=True, parents=True)

rebuild_col = False

if rebuild_col:
    opt_collision = make_planner_with_collision(
        robots,
        carried_object,
        num_samples,
        num_control_points,
        order,
        diff_co_options,
        gaze_options,
        collision_options,
        opt_with_collision_compile_path,
    )
else:
    opt_collision = load_opt_info(
        opt_with_collision_compile_path,
        robots,
        carried_object,
        num_control_points,
        order,
        diff_co_options,
        gaze_options,
        collision_options,
    )
opt_collision_very_short = load_opt_info(
    opt_with_collision_compile_path,
    robots,
    carried_object,
    num_control_points,
    order,
    diff_co_options, 
    gaze_options,
    collision_options_very_short,
)


In [ ]:
aaa

In [ ]:

# import threadpoolctl
# with threadpoolctl.threadpool_limits(limits=12):
#     time_function(opt_collision.nlp_hess_l)
#     time_function(opt_collision.nlp_jac_g)
#     time_function(opt_collision.nlp_g)
#     time_function(opt_collision.nlp_f)
#     time_function(opt_collision.nlp_grad_f)

In [ ]:
from diff_co_lcm import lcmt_gaze_polytopes,lcmt_pose
import mpc.gaze_functions as gaze


def meshcat_arrow(meshcat, path,position, direction, size, head_size ):
    p1 = np.zeros((3,1))
    x_axis = np.array([1.,0,0]).reshape(-1,1)
    y_axis = np.array([0,1.,0]).reshape(-1,1)
    z_axis = np.array([0,0,1.]).reshape(-1,1)
    rotation_matrix = RotationMatrix.MakeFromOneVector(direction,0)
    transform = RigidTransform(p=position, R=rotation_matrix)
    p2 = p1 + x_axis*size
    line_head_1_a = p1 + y_axis*head_size + x_axis*(size-head_size)
    line_head_1_b = p1 + z_axis*head_size + x_axis*(size-head_size)
    line_head_2_a = p1 - y_axis*head_size + x_axis*(size-head_size)
    line_head_2_b = p1 - z_axis*head_size + x_axis*(size-head_size)
    start = p1
    end = p2
    meshcat.SetLineSegments(path,start,end,1)
    vertices = np.hstack([p2,line_head_1_b,line_head_2_a,line_head_2_b,line_head_1_a])
    faces = np.array([[0,1,2],[0,2,3],[0,3,4],[0,4,1],[1,2,3],[1,3,4]]).T
    meshcat.SetTriangleMesh(path+"/head",vertices,faces)
    meshcat.SetTransform(path,transform)
def my_handler(*args):
    global polytopes,line_point_,transform_masks,ik_mask,iks_robot_1,polyharmonic_weights,polyharmonic_sv,polynomial_weights
    # print(args[0].decode())
    t0  = time.perf_counter()
    msg = lcmt_gaze_polytopes.decode(args[0])
    
    vertices = np.asarray(msg.vertices)
    simplices = np.asarray(msg.simplices)
    indices = np.asarray(msg.indices)
    for i in range(30):
        meshcat.Delete(f'/polytope_{i}')
    polytopes  = []

    placing_spot_pose = RigidTransform(planner.lcm_subscription_handler.placing_spot_pose)

    pose_gaze_samples,iks_robot_1,transform_masks, ik_mask = gaze.get_samples_around_placing_spot(sampling_size_around_spot,num_samples_spot,num_samples_gaze,num_samples_gaze_turn,placing_spot_pose,q7_guesses,elevation,robot_base_pose_1)
    line_point_ = pose_gaze_samples[:,:3,3].astype(np.float64)
    line_direction_ = pose_gaze_samples[:,:3,2].astype(np.float64)
    for i,(a,b,c,d) in enumerate(indices):
        polytope_vertices= vertices[a:b]

        polytope_simplices = simplices[c:d]
        meshcat.SetTriangleMesh(f'/polytope_{i}',polytope_vertices.T,polytope_simplices.T)

        polytopes.append( { 'vertices': vertices[a:b], 'simplices': simplices[c:d] } )
        mean_vertices = np.mean(polytope_vertices,axis=0)
        if np.linalg.norm(mean_vertices-placing_spot_pose.translation()) > 0.5:
            continue
        polytope_vertices_ = polytope_vertices.astype(np.float64)
        faces = polytope_vertices[polytope_simplices]
        faces_ = np.array(faces).astype(np.float64)



        intersects = gaze.line_intersects_with_polytope_batch(polytope_vertices_,faces_, line_point_[transform_masks.squeeze()], line_direction_[transform_masks.squeeze()])
        intersects_ = np.ones(transform_masks.shape[0],dtype=bool)
        intersects_[transform_masks.squeeze()] = intersects
        transform_masks = np.logical_and(transform_masks,~intersects_.reshape(-1,1))

    where_not_nan = np.nonzero(ik_mask[transform_masks.squeeze()])
    unique_rows, indices = np.unique(where_not_nan[0], return_index=True)
    unique_gazes = iks_robot_1[transform_masks.squeeze()][where_not_nan][indices]
    support_vectors = torch.from_numpy(unique_gazes).to(torch.float32)
    Y_support_vectors  = torch.ones(unique_gazes.shape[0])
# robots[0].vision_model.forward_kinematics_groups_torch['group_1']
# 
    # fk_support_vectors = robots[0].vision_model.forward_kinematics_groups_torch['group_1'](support_vectors).squeeze()
    fk_sv, K_rg = kernel_rg(support_vectors,2.,0.1)
    fk_sv = fk_sv.view(support_vectors.shape[0],-1)
    # mock_weights = torch.ones((fk_sv.shape[0],1))
    # Y_support_vectors = torch.vmap(lambda sv,svv, w: kernel_gaze(sv,svv)@w, in_dims = (0,None,None))(fk_support_vectors.view(fk_support_vectors.shape[0],-1),fk_support_vectors.view(fk_support_vectors.shape[0],-1),mock_weights)
    Y_support_vectors = K_rg.sum(dim=1)
    Y_support_vectors /= (Y_support_vectors).max()
    # Y_support_vectors = torch.ones(Y_support_vectors.shape[0])
    _, K_ph = kernel_ph(support_vectors,1,None)
    # print(fk_sv[0],Y_support_vectors[0])
    # polyharmonic_weights = torch.linalg.solve(K_ph, Y_support_vectors.to(torch.float32))
    N, d = fk_sv.shape
    B = torch.cat([torch.ones((N, 1),device=support_vectors.device), fk_sv], dim=1)
    top = torch.cat([K_ph, B], dim=1)
    bottom = torch.cat([B.T, torch.zeros((d+1, d+1), device=support_vectors.device)], dim=1)
    M = torch.cat([top, bottom], dim=0)
    f = torch.cat([Y_support_vectors.view(-1,1), torch.zeros((d+1, 1), device=support_vectors.device)])
    try:
        polyharmonic_weights = torch.linalg.solve(M, f)
    except:
        print('solve failed, doing pseudoinverse')
        M_inv = torch.linalg.pinv(M)
        polyharmonic_weights = M_inv @ f
    support_vectors = support_vectors.to(torch.float64)
    # K_ph = torch.vmap(polyharmonic_kernel, in_dims = (0,None))(fk_support_vectors.view(fk_support_vectors.shape[0],-1),fk_support_vectors.view(fk_support_vectors.shape[0],-1))
    # polyharmonic_weights = torch.linalg.solve(K_ph, Y_support_vectors.to(torch.float32))
    # polyharmonic_weights = polyharmonic_weights - polyharmonic_weights.mean()
    polyharmonic_sv = fk_sv
    print('time',time.perf_counter()-t0)
    is_placing_spot_hidden = msg.is_placing_spot_hidden
    num_support_vectors = support_vectors.shape[0]
    polynomial_weights = polyharmonic_weights[num_support_vectors :].to(torch.float64)
    polyharmonic_weights = polyharmonic_weights[:num_support_vectors].to(torch.float64)
    # q1 = support_vectors[0]
    # (fk_S,fk_SV), K = kernel_ph([(q1).view(1,-1),support_vectors],1,None)
    # print(K,polyharmonic_weights,polynomial_weights,fk_S)
    # score = (K.to(torch.float64)@polyharmonic_weights.view(-1,1)).view(-1) + polynomial_weights[0].reshape(-1).to(torch.float64) + torch.dot(polynomial_weights[1:].reshape(-1).to(torch.float64), fk_S.reshape(-1).to(torch.float64))
    # print(fk_S,score)
    # asdfdfssd
    # for i in range(2000): meshcat.Delete(f"/gaze/gaze_direction_{i}")
    # for i in range(2000): meshcat.Delete(f"/gaze/gaze_direction_camera_{i}")
    # for i in range(2000): meshcat.Delete(f"/gaze/gaze_direction_gradient_{i}")
    # for i in range(2000): meshcat.Delete(f"/gaze/gaze_direction_gradient_{i}_2")
    # for i in range(2000): meshcat.Delete(f"/gaze/gaze_direction_gradient_{i}_line")
    # for i in range(2000): meshcat.Delete(f"/gaze/gaze_direction_gradient_{i}")
    # for i in range(2000): meshcat.Delete(f"/gaze/gaze_direction_gradient_{i}_b")
    # for j in range(0,100):
    #     print('j',j)
    #     print()
    #     print()
    #     print()
    #     print()
    #     for i,q1, in enumerate(unique_gazes):
    #         if i+j*unique_gazes.shape[0] > 200:
    #             print("max samples reached")
    #             break
    #         q1 = q1 + np.random.randn(*q1.shape)*(j/20+0.1)
    #         q1 = np.concatenate([q1,np.zeros(2)])
    #         # viz_helper.set_position('robot_0',q1)
    #         # fk = robots[0].forward_kinematic_gaze(q1).full()
    #         fk = robots[0].vision_model.forward_kinematics_groups_casadi['group_1'](q1).full()
    #         z = fk[0,:] - fk[1,:]
    #         rotation = RotationMatrix.MakeFromOneVector(z,2)
            
    # # def svm_callback(self, msg, robot_name, group_name):

    # #     msg = lcmt_support_vector.decode(msg)

    # #     support_vectors_1 = msg.support_vectors
    # #     weights_1 = msg.weights[: msg.num_support_vectors]
    # #     # constant_1 = msg.weights_1[msg.num_support_vectors_1]
    # #     # pol_weights_1 = msg.weights_1[msg.num_support_vectors_1 + 1 :]
    # #     pol_weights_1 = msg.weights[msg.num_support_vectors :]
    # #     # print(msg.weights)
    # #     sv_1 = np.array(support_vectors_1).T
    # #     w_1 = np.array(weights_1).T
    # #     # constant_1 = np.array(constant_1).T
    # #     pols_1 = np.array(pol_weights_1).T
    # #     self.last_svm[robot_name][group_name].append(
    # #         SVM(weights=w_1, sv=sv_1, pol_weights=pols_1)
    # #     )

    #         # v = torch.from_numpy(fk).to(torch.float32).view(-1)
    #         # v.requires_grad = True
    #         # score = polyharmonic_kernel(v,polyharmonic_sv.view(polyharmonic_sv.shape[0],-1))@polyharmonic_weights
    #         # score = kernel_ph([torch.from_numpy(q1),polyharmonic_sv],2,None)[1]@polyharmonic_weights
    #         # score = kernel_ph([torch.from_numpy(q1).view(1,-1),support_vectors],2,None)[1]@polyharmonic_weights
    #         (fk_S,fk_SV), K = kernel_ph([torch.from_numpy(q1).view(1,-1),support_vectors],2,None)
    #         fk_S.requires_grad = True
    #         score = (K.to(torch.float64)@polyharmonic_weights.view(-1,1)).view(-1) + polynomial_weights[0].reshape(-1).to(torch.float64) + torch.dot(polynomial_weights[1:].reshape(-1).to(torch.float64), fk_S.reshape(-1).to(torch.float64))
    #         print(score)
    #         sample_index = i+j*unique_gazes.shape[0]
            
    #         # if score > 0:
    #         #     meshcat.SetObject(f"/gaze/gaze_direction_gradient_{sample_index}",Sphere(score*0.01))
    #         #     meshcat.SetProperty(f"/gaze/gaze_direction_gradient_{sample_index}",'position',fk_S.view(-1,3)[0].detach().numpy())
    #         #     meshcat.SetObject(f"/gaze/gaze_direction_gradient_{sample_index}_2",Sphere(score*0.01))
    #         #     meshcat.SetProperty(f"/gaze/gaze_direction_gradient_{sample_index}_2",'position',fk_S.view(-1,3)[1].detach().numpy())
    #         #     meshcat.SetProperty(f"/gaze/gaze_direction_gradient_{sample_index}", 'color', [1,0,0,1])
    #         # continue
    #         score.backward()


    #         gradient = fk_S.grad.view(-1,3)
    #         direction_EE = gradient[1]
    #         arrow_size_EE = torch.linalg.norm(gradient[1])/40
    #         direction_8 = gradient[0]
    #         arrow_size_8 = torch.linalg.norm(gradient[0])/40
            

            
    #         meshcat_arrow(meshcat,f"/gaze/gaze_direction_gradient_{sample_index}",fk[1,:],direction_EE.numpy(),arrow_size_EE.numpy(),arrow_size_EE.numpy()/10)
    #         meshcat_arrow(meshcat,f"/gaze/gaze_direction_gradient_{sample_index}_2",fk[1,:] + 0.05*(fk[0,:] - fk[1,:]),direction_8.numpy(),arrow_size_8.numpy(),arrow_size_8.numpy()/10)
    #         meshcat.SetLine(f'/gaze/gaze_direction_gradient_{sample_index}_line',np.hstack([fk[1,:].reshape(3,1), (fk[1,:] + 0.05*(fk[0,:] - fk[1,:])).reshape(3,1)]))
    #         time.sleep(0.002)
    #         meshcat.SetProperty(f"/gaze/gaze_direction_gradient_{sample_index}", 'color', [1,0,0,1])
    #         meshcat.SetProperty(f"/gaze/gaze_direction_gradient_{sample_index}_2", 'color', [0,1,0,1])
    #         meshcat.Flush()
    #         # viz_helper.publish()
    #         time.sleep(0.01)
    
    # if is_placing_spot_hidden:
    #     meshcat.SetProperty(f'/placing_spot','color',[1,0,0,1])
    # else:
    #     meshcat.SetProperty(f'/placing_spot','color',[0,1,0,1])

    # for i,(pose, has_ik) in enumerate(zip(pose_gaze_samples, transform_masks)):

    #     rotation = RotationMatrix(pose[:3,:3])
    #     position = pose[:3,3]
    #     meshcat.SetProperty(f'/samples/sample_{i}','visibility',True)
    #     meshcat.SetTransform(f"/samples/sample_{i}", RigidTransform(p=position,R = rotation))
    #     if has_ik:
    #         meshcat.SetProperty(f"/samples/sample_{i}", "color",[0,1,0,0.5])
    #     else:
    #         meshcat.SetProperty(f"/samples/sample_{i}", "color",[1,1,1,0.5])
    # for i in range(600-i):
    #     meshcat.SetProperty(f'/samples/sample_{i}','visibility',False)
    # dagdsgds

# kernel_gaze = ForwardKinematicKernel(RationalQuadraticKernel(alpha = 2,length_scale = 0.05),robots[0].forward_kinematic_gaze)


from pydrake.all import DrakeLcm
lcm = DrakeLcm()
import pydrake.geometry as pydgeo
COMMAND_HZ = 20
lcm = DrakeLcm()
subscription = lcm.Subscribe("gaze_polytopes", my_handler)
polytopes = []
sampling_size_around_spot = 0.01
num_samples_spot = 50
num_samples_gaze = 2 #for each spot sample
num_samples_gaze_turn = 1 #for each gaze sample, rotate around z axis, set to one to not make K_ph singular
num_q7s = 7
q7_guesses = np.linspace(robots[0].plant.GetPositionLowerLimits()[6],robots[0].plant.GetPositionUpperLimits()[6],num_q7s)
robot_base_pose_1 = np.linalg.inv(robots[0].plant.GetFrameByName('panda_link0').CalcPose(robots[0].plant.CreateDefaultContext(),robots[0].plant.GetFrameByName('world')).GetAsMatrix4())
placing_spot_pose = RigidTransform(p=[0.1, -0.4, 0.0],rpy=RollPitchYaw([0, 0, np.pi/4]))
elevation = [0.5,0.55]
from diff_co.geometrical_model import Kernel
kernel_rg = Kernel(robots[0].vision_model.forward_kinematics_groups_torch['group_1'],'','rational_quadratic')
kernel_ph = Kernel(robots[0].vision_model.forward_kinematics_groups_torch['group_1'],'','polyharmonic')
# subscription
lcm.HandleSubscriptions(0)
# try:
#     while True:
#         # lcm.Publish(channel="camera_pose", buffer = state.encode())
#         lcm.HandleSubscriptions(0)
#         time.sleep(1)
# except KeyboardInterrupt:
#     pass

In [ ]:
lcm.HandleSubscriptions(0)

In [ ]:
import mpc.planner_implementation.lcm_handler
import mpc.planner_implementation.ros_handler
import mpc.planner_implementation.helper
import mpc.planner_implementation.helper_mixin
import mpc.helper_functions
importlib.reload(mpc.planner_implementation.lcm_handler)
importlib.reload(mpc.planner_implementation.ros_handler)
importlib.reload(mpc.planner_implementation.helper_mixin)
importlib.reload(mpc.planner_implementation.helper)
importlib.reload(mpc.helper_functions)
from mpc.planner_implementation.helper import Planner
from mpc.planner_implementation.lcm_handler import VisionParameters
from mpc.planner_implementation.ros_handler import ROSHandler
ros_handler = ROSHandler(None)
lcm_subscription_handler_configuration = VisionParameters(
    sampling_size_around_spot = 1,
    num_samples_spot=1,
    num_samples_gaze=1,
    num_samples_gaze_turn=1,
    q7_guesses=1,
    elevation=1,
    robot_base_pose_1= 1,
    polyharmonic_kernel = None,
    kernel_gaze = None,
)
planner = Planner(opt_collision, 
                  meshcat,
                    lcm_subscription_handler_configuration = lcm_subscription_handler_configuration, 
                    ros_handler = ros_handler, 
                    viz_helper = viz_helper,
                    num_parallel_plans = 12
)
planner.set_plan_grasping_bounds(np.array([0.00,0.1]),np.array([-0.1,-0.00]))
planner.set_costs(vision_cost = 0, duration_cost = 0.001, acceleration_cost = 0.001, manipulability_cost = .0001, replan_connection_cost = 1.,slack_cost_weight = -0.1)
planner.set_time_bounds_plan(0.5,20.)
planner.set_velocity_scaling(0.1)
planner.parallel_solve_data.set_initial_guess('duration', np.array(2.))
planner.initial_plan_data.set_initial_guess('duration', np.array(2.))

planner.EE_transform_1 = RigidTransform(
    p=[0.1, 0, 0], rpy=RollPitchYaw([0, 0, 0])
).GetAsMatrix4()
planner.EE_transform_2 = RigidTransform(
    p=[-0.1, 0, 0], rpy=RollPitchYaw([0, 0, np.pi])
).GetAsMatrix4()


### Parallel solves to get the initial guesses

In [ ]:
def set_parallel_initial_guesses(self,num_q7s,initial_pose,end_pose):
    num_initial_guesses = self.parallel_solve_data.map_size
    q1 = self.robot_1_inverse_kinematics.casadi_obj_IK_compiled(initial_pose,np.linspace(-np.pi,np.pi,num_q7s).reshape(1,-1),np.array([0.]*7),self.EE_transform_1)
    q1 = q1.full().reshape(-1,7)
    q1 = q1[~np.isnan(q1).any(axis=1)]

    q2 = self.robot_2_inverse_kinematics.casadi_obj_IK_compiled(initial_pose,np.linspace(-np.pi,np.pi,num_q7s).reshape(1,-1),np.array([0.]*7),self.EE_transform_2)
    q2 = q2.full().reshape(-1,7)
    q2 = q2[~np.isnan(q2).any(axis=1)]
    q1_combinations = q1[np.arange(q1.shape[0]).repeat(q2.shape[0])]
    q2_combinations = q2[np.tile(np.arange(q2.shape[0]),q1.shape[0])]
    indices = np.random.randint(0,q2_combinations.shape[0],num_initial_guesses)
    q_1_start = q1_combinations[indices]
    q_2_start = q2_combinations[indices]
    assert q_1_start.size > 0
    assert q_2_start.size > 0
    obj_start = np.concatenate((RotationMatrix(initial_pose[:3,:3]).ToQuaternion().wxyz(),initial_pose[:3,3]  + np.array([0,0,0.1])))
    obj_end = np.concatenate((RotationMatrix(end_pose[:3,:3]).ToQuaternion().wxyz(),end_pose[:3,3] + np.array([0,0,0.1])))
    self.parallel_solve_data.set_initial_guess('carried_object_control_points', np.linspace(obj_start,obj_end,self.num_control_points_mpc).T)
    for i,(q1,q2) in enumerate(zip(q_1_start,q_2_start)):
        self.parallel_solve_data.set_initial_guess('robot_1_control_points', configuration_to_control_points(q1, self.num_control_points_mpc),i)
        self.parallel_solve_data.set_initial_guess('robot_2_control_points', configuration_to_control_points(q2, self.num_control_points_mpc),i)
planner.set_parallel_initial_guesses = partial(set_parallel_initial_guesses,planner)




planner.parallel_solve_data.x0[:] = 0.
planner.parallel_solve_data.lam_g0[:] = 0.
planner.parallel_solve_data.lam_x0[:] = 0.
num_q7s = 20
planner.parallel_solve_data.set_initial_guess('duration', np.array(2.))
planner.set_parallel_initial_guesses(num_q7s,planner.lcm_subscription_handler.initial_pose,planner.lcm_subscription_handler.placing_spot_pose)
planner.set_collision_bounds(-np.inf,np.inf)


with threadpoolctl.threadpool_limits(limits={'blas':1,'openmp':12}):
    planner.parallel_plans()
# print('hi')
    # planner.parallel_plans(q_1_start,q_2_start)
    # planner.parallel_plans(q_1_start,q_2_start)

    
planner.set_collision_bounds(-np.inf,0.4)
solved = []
for i in range(planner.parallel_solve_data.map_size):
    if planner.parallel_solve_data.within_bounds()[i]:
        solved.append(i)
not_solved = [i for i in range(planner.parallel_solve_data.map_size) if i not in solved]
for i in not_solved:
    # choose one of the solved indices
    j = np.random.choice(solved)
    noisified_control_points = planner.parallel_solve_data.result['robot_1_control_points'][j] + np.random.normal(0,0.1,planner.parallel_solve_data.result['robot_1_control_points'][j].shape)
    planner.parallel_solve_data.set_initial_guess('robot_1_control_points',noisified_control_points ,i)
    noisified_control_points = planner.parallel_solve_data.result['robot_2_control_points'][j] + np.random.normal(0,0.1,planner.parallel_solve_data.result['robot_2_control_points'][j].shape)
    planner.parallel_solve_data.set_initial_guess('robot_2_control_points',noisified_control_points ,i)
    planner.parallel_solve_data.lam_g0[:,i] = 0.
    planner.parallel_solve_data.lam_x0[:,i] = 0.
with threadpoolctl.threadpool_limits(limits={'blas':1,'openmp':12}):
    planner.parallel_plans()
# planner.set_collision_bounds(-np.inf,0.2)
# with threadpoolctl.threadpool_limits(limits={'blas':1,'openmp':12}):
#     planner.parallel_plans()
# planner.set_collision_bounds(-np.inf,0.0)
# with threadpoolctl.threadpool_limits(limits={'blas':1,'openmp':12}):
#     planner.parallel_plans()
#     planner.parallel_plans()
# point_cloud_viz.update()


In [ ]:
meshcat.web_url()

### How's the score of each of those solves, display_initial_guesses shows them in meshcat. Also print how the constraints are being violated, to show which of the previous solves are actually feasible-ish.

In [ ]:
point_cloud_viz.update()
i = 0
bspline_1_col = (BSpline(planner.parallel_solve_data.result['robot_1_control_points'][i],opt_collision.order))
bspline_2_col = (BSpline(planner.parallel_solve_data.result['robot_2_control_points'][i],opt_collision.order))
bspline_obj_col = (BSpline(planner.parallel_solve_data.result['carried_object_control_points'][i],opt_collision.order))
# planner.initial_plan_data.x[:] = planner.parallel_solve_data.x[:,i].reshape(-1,1)
display_initial_guesses(planner.parallel_solve_data,True)
# print_collision_score_on_samples(30,planner,bspline_1_col,bspline_2_col)
# publish_trajectory(planner.initial_plan_data, planner.lcm_subscription_handler, order = 3)
# collision_score_from_parallel_solves(30,planner)
print([np.sum(x+y) for x,y in zip(planner.parallel_solve_data.ubg_violation(),planner.parallel_solve_data.lbg_violation())])
# print()
collision_score_from_parallel_solves(30,planner)


In [11]:
# end_position_lower_bounds = (
#     -np.ones(
#         3,
#     )
#     * 1.5
# )
# end_position_upper_bounds = (
#     np.ones(
#         3,
#     )
#     * 1.5
# )
# end_angle_bounds = np.array([np.pi])

# # polyharmonic_weights,polyharmonic_sv,polynomial_weights
# # lcm.HandleSubscriptions(0)
# planner.set_costs(vision_cost = 1000., duration_cost = 0.1, acceleration_cost = 0.01, manipulability_cost = .0001,replan_connection_cost = 100.,slack_cost_weight = -100.)

# vision_weight = planner.vision_weight
# polyharmonic_sv_np = polyharmonic_sv.numpy().reshape(polyharmonic_sv.shape[0],-1).T
# polyharmonic_weights_np = polyharmonic_weights.numpy().T
# polynomial_weights_np = polynomial_weights.numpy()
# name = f'robot_1_group_1_vision'
# for opt_data in [planner.initial_plan_data, planner.parallel_solve_data, planner.replan_data]:
#     weights = np.zeros(opt_data.parameter_shapes[name + '_weights'])
#     support_vectors = np.zeros(opt_data.parameter_shapes[name + '_support_vectors'])
#     weights[:, :polyharmonic_weights_np.shape[1]] = polyharmonic_weights_np.reshape(1,-1)
#     support_vectors[:, :polyharmonic_sv_np.shape[1]] = polyharmonic_sv_np
#     polynomial_weights_ = np.zeros(opt_data.parameter_shapes[name + '_polynomial_weights'])
#     ww = polynomial_weights_np.reshape(-1,1)
#     polynomial_weights_[:ww.shape[0]] = ww
#     opt_data.set_parameter('vision_cost_weight',np.array(vision_weight))
#     opt_data.set_parameter('robot_1_group_1_vision_polynomial_weights', polynomial_weights_)
#     opt_data.set_parameter('robot_1_group_1_vision_weights', weights)
#     opt_data.set_parameter('robot_1_group_1_vision_support_vectors', support_vectors)
    
#     opt_data.set_parameter(
#         "vision_cost_weight", np.array(vision_weight))
#     opt_data.set_parameter(
#         "end_position_lower_bounds",
#         end_position_lower_bounds,
#     )
#     opt_data.set_parameter(
#         "end_position_upper_bounds",
#         end_position_upper_bounds,
#     )
#     opt_data.set_parameter(
#     "end_angle_bounds",
#     end_angle_bounds,
# )

In [ ]:
lcm.HandleSubscriptions(0)

In [ ]:
meshcat.SetObject(f'/placing_spot',Sphere(0.05))
meshcat.SetProperty(f'/placing_spot','color',[0,1,0,1])
meshcat.SetProperty(f'/placing_spot','visibility',True)
meshcat.SetTransform(f"/placing_spot", RigidTransform(planner.lcm_subscription_handler.placing_spot_pose))

### Solve one trajectory defined by index (done manually, usually just pick the one with least collision or most feasible among the ones calculated in parallel)


In [ ]:

end_position_lower_bounds = (
    -np.ones(
        3, 
    )
    * 1.5
)
end_position_upper_bounds = (
    np.ones(
        3,
    )
    * 1.5
)
end_angle_bounds = np.array([np.pi])
planner.set_velocity_scaling(0.1)
point_cloud_viz.update()
planner.set_costs(vision_cost = -1000., duration_cost = 0.1, acceleration_cost = 0.01, manipulability_cost = .0001,replan_connection_cost = 100.,slack_cost_weight = -100.)
index = 3
planner.initial_plan_data.x0[:] = planner.parallel_solve_data.x[:,index].reshape(-1,1)
planner.initial_plan_data.lam_g0[:] = planner.parallel_solve_data.lam_g[:,index].reshape(-1,1)
planner.initial_plan_data.lam_x0[:] = planner.parallel_solve_data.lam_x[:,index].reshape(-1,1)
planner.initial_plan_data.p[:] = planner.parallel_solve_data.p[:,index].reshape(-1,1)
planner.set_collision_bounds(-np.inf,-.0)
vision_weight = planner.vision_weight
polyharmonic_sv_np = polyharmonic_sv.numpy().reshape(polyharmonic_sv.shape[0],-1).T
polyharmonic_weights_np = polyharmonic_weights.numpy().T
polynomial_weights_np = polynomial_weights.numpy()
name = f'robot_1_group_1_vision'
for opt_data in [planner.initial_plan_data, planner.parallel_solve_data, planner.replan_data]:
    weights = np.zeros(opt_data.parameter_shapes[name + '_weights'])
    support_vectors = np.zeros(opt_data.parameter_shapes[name + '_support_vectors'])
    weights[:, :polyharmonic_weights_np.shape[1]] = polyharmonic_weights_np.reshape(1,-1)
    support_vectors[:, :polyharmonic_sv_np.shape[1]] = polyharmonic_sv_np
    polynomial_weights_ = np.zeros(opt_data.parameter_shapes[name + '_polynomial_weights'])
    ww = polynomial_weights_np.reshape(-1,1)
    polynomial_weights_[:ww.shape[0]] = ww
    opt_data.set_parameter('vision_cost_weight',np.array(vision_weight))
    opt_data.set_parameter('robot_1_group_1_vision_polynomial_weights', polynomial_weights_)
    opt_data.set_parameter('robot_1_group_1_vision_weights', weights)
    opt_data.set_parameter('robot_1_group_1_vision_support_vectors', support_vectors)
    
    opt_data.set_parameter(
        "vision_cost_weight", np.array(vision_weight))
    opt_data.set_parameter(
        "end_position_lower_bounds",
        end_position_lower_bounds*0,
    )
    opt_data.set_parameter(
        "end_position_upper_bounds",
        end_position_upper_bounds*0,
    )
    opt_data.set_parameter(
    "end_angle_bounds",
    end_angle_bounds*0,
)
import threadpoolctl
with threadpoolctl.threadpool_limits(limits={'blas':1,'openmp':12}):
    planner.initial_plan()
    

display_trajectory(planner.initial_plan_data,meshcat)
planner.initial_plan_data.print_violated_g()


### Print the vision score along the trajectory for debugging, also sends the trajectory to the learning algorithm to sample more along the calculated path. It might be wise to recalculate the previous trajectory again using this new finer sampling (just rerun the previous cell, the support vectors are read anew in planner.initial_plan()), the first time you run publish_trajectory the svm_learning* program will recompile some stuff, so that might take a minute or two. So wait a minute or so before rerunning the previous cell (only first time after starting the learning notebook)

In [ ]:
def print_vision_score_on_samples(num_samples,planner,bspline_1,bspline_2):
    def polyharmonic_kernel(
        x: torch.Tensor, y: torch.Tensor, alpha: int
    ) -> torch.Tensor:

        if alpha % 2 == 1:
            return torch.linalg.norm(x - y, axis=-1) ** alpha
        else:
            r = torch.linalg.norm(x - y, axis=-1)
            temp = (r**alpha) * torch.log(r)
            temp[torch.isnan(temp)] = 0.0

            return temp
        
    for s in np.linspace(0,1,num_samples):
        print(s)
        text= ''
        for robot in ['robot_1']:
            for group_name in ['group_1',]:
                # svm = planner.lcm_subscription_handler.last_svm[robot][group_name][-1]
            # (fk_S,fk_SV), K = kernel_ph([torch.from_numpy(q1).view(1,-1),support_vectors],2,None)
            # fk_S.requires_grad = True
            # score = (K.to(torch.float64)@polyharmonic_weights.view(-1,1)).view(-1) + polynomial_weights[0].reshape(-1).to(torch.float64) + torch.dot(polynomial_weights[1:].reshape(-1).to(torch.float64), fk_S.reshape(-1).to(torch.float64))

                weights = torch.as_tensor(polyharmonic_weights_np)
                support_vectors = torch.as_tensor(polyharmonic_sv_np)
                polynomial_weights = torch.as_tensor(polynomial_weights_np).squeeze()
                q = torch.as_tensor(bspline_2.evaluate(s) if robot == 'robot_2' else bspline_1.evaluate(s))
                # col_model = planner.robot_1_collision_model if robot == 'robot_1' else planner.robot_2_collision_model
                # q = support_vectors
                fk_q = robots[0].vision_model.forward_kinematics_groups_torch[group_name](q)

                sample = fk_q.reshape(-1)
                # sample = support_vectors[:,0]
                # print(sample)
                dist = polyharmonic_kernel(sample.view(1,-1), support_vectors.T, 1)
                score = torch.dot(weights.reshape(-1).to(torch.float64), dist.reshape(-1).to(torch.float64)) + polynomial_weights[0].reshape(-1).to(torch.float64) + torch.dot(polynomial_weights[1:].reshape(-1).to(torch.float64), sample.reshape(-1).to(torch.float64))
                # print(s,score)
                # print(dist,weights,polynomial_weights,sample)
                text += f'{group_name}: {score.item():.2f} '
            text+= '\n'
        print(text)
bspline_1_col = (BSpline(planner.initial_plan_data.result['robot_1_control_points'],opt_collision.order))
bspline_2_col = (BSpline(planner.initial_plan_data.result['robot_2_control_points'],opt_collision.order))
publish_trajectory(planner.initial_plan_data, planner.lcm_subscription_handler, order = 3)
print_vision_score_on_samples(20,planner,bspline_1_col,bspline_2_col)
# %%prun
# bspline_obj_col = (BSpline(planner.initial_plan_data.result['carried_object_control_points'],opt_collision.order))
# print_collision_score_on_samples(10,planner,bspline_1_col,bspline_2_col)

# v = robots[1].collision_model.robot_geometry_functions_torch['panda_hand'](torch.as_tensor(bspline_2_col.evaluate(0)))
# for i,vv in enumerate(v):
#     meshcat.SetObject(f'/teste{i}',Sphere(0.05))
#     meshcat.SetProperty(f'/teste{i}','position',vv.numpy())
    # tensor([0.0265, 0.4373, 0.5390, 0.0159, 0.3620, 0.4981]) tensor(0.7499)
# polyharmonic_sv_np[:,0]


In [ ]:
# robots[0].vision_model.forward_kinematics_groups_torch['group_1'](torch.tensor(bspline_1_col.evaluate(1)))

In [ ]:
# robots[0].vision_model.forward_kinematics_groups_torch

In [ ]:
# polynomial_weights_np

In [ ]:
# polyharmonic_sv_np.shape

### Redefining some functions from the planner

In [ ]:
from multiprocessing import shared_memory
from multiprocessing import Process, resource_tracker
def stop_recording_simulation(self):
    q_1s = np.concatenate(self.q_1s,axis= 0 )
    q_2s = np.concatenate(self.q_2s,axis= 0 )
    q_objs = np.concatenate(self.q_obj,axis= 0 )
    ts = np.concatenate(self.ts,axis= 0 )
    dt = self.ts[0][-1] - self.ts[0][0]
    self.meshcat.StartRecording(frames_per_second=30.0)
    k = 0
    self.meshcat.Delete('/obstacles')
    obstacle_radii = 0.04
    # self.meshcat.Delete(f"/obstacles/{i}")
    # for i in range(500):
    for i in range(1000):
        # for i, q_sample in enumerate(point_clo ud):
        self.meshcat.SetObject(
            f"/obstacles/{i}", Sphere(obstacle_radii)
        )
        self.meshcat.SetProperty(
            f"/obstacles/{i}", "position", [0,0,0], time_in_recording = 0,
        )
    for i in range(q_1s.shape[0]):
        t = ts[i] - ts[0]
        if t - k*dt > 0:
            k += 1
            try:
                obstacles = self.obstacles[k]
                obstacles = obstacles[~np.isnan(obstacles).any(axis=1)]
                # print(obstacles.shape)
                for j in range(obstacles.shape[0]):
                    self.meshcat.SetProperty(
                            f"/obstacles/{j}", "position", obstacles[j], time_in_recording = t - ts[0],
                        )
                for j in range(obstacles.shape[0],1000):
                    self.meshcat.SetProperty(
                            f"/obstacles/{j}", "position", [0,0,0], time_in_recording = t - ts[0],
                        )
            except:
                print(k)
        q_1 = q_1s[i]
        q_1 = np.concatenate([q_1,np.zeros(2)])

        q_2 = q_2s[i]
        q_2 = np.concatenate([q_2,np.zeros(2)])
        q_obj = q_objs[i]
        self.viz_helper.set_position('robot_0',q_1)
        self.viz_helper.set_position('robot_1',q_2)
        self.viz_helper.set_position('carried_object',q_obj)
        self.viz_helper.diagram_context.SetTime(ts[i] - ts[0])
        self.viz_helper.publish()
    for i,q_obj in enumerate(self.q_obj):
        q_first = q_obj[0]
        q_last = q_obj[-1]
        self.meshcat.SetObject(
            f'/first_point/{i}', Sphere(0.01)
        )
        self.meshcat.SetProperty(
            f'/first_point/{i}', 'position', q_first[4:]
        )
        # self.meshcat.SetProperty(f"thing/{i}", "color",[1,0,0,1])
        self.meshcat.SetProperty(f"/first_point/{i}", "color", [1, 0, 0, 0.5])
        # print( q_first[3:])
        self.meshcat.SetObject(
            f'/last_point/{i}', Sphere(0.01)
        )
        self.meshcat.SetProperty(
            f'/last_point/{i}', 'position', q_last[4:]
        )
        self.meshcat.SetProperty(f"/last_point/{i}", "color", [0,1, 0, 0.5])
    self.meshcat.StopRecording()
    self.meshcat.PublishRecording()
def start_recording_simulation(self):
    self.q_1s = []
    self.q_2s = []
    self.q_obj = []
    self.obstacles = []
    self.ts = []
    self.meshcat.StartRecording(frames_per_second=30.0)
planner.start_recording_simulation = partial(start_recording_simulation,planner)
planner.stop_recording_simulation = partial(stop_recording_simulation,planner)
def follow_pointwise_trajectory(self,q_1s,q_2s,q_1_dots,q_2_dots,q_1_dotdots,q_2_dotdots, Ts, start_time):
    # if self.simulation:
    #     for q_1,q_1_dot,q_2,q_2_dot,t  in zip(q_1s,q_2_dots,q_1s,q_2_dots,np.linspace(0,time,num_steps)):
    #         self.viz_helper.set_position('robot_1',q_1)
    #         self.viz_helper.set_position('robot_2',q_2)
    #         self.viz_helper.diagram_context.SetTime(self.simulation_time + t)
    #         self.viz_helper.publish()
    #     self.simulation_time += time
    #     return 
    # http://wiki.ros.org/joint_trajectory_controller/UnderstandingTrajectoryReplacement
    goal_1 = FollowJointTrajectoryGoal()
    goal_1.trajectory.joint_names = self.JOINT_NAMES
    num_steps = len(q_1s)
    tol = ros.Duration.from_sec(0.01)
    t0 = 0.00
    Ts += t0
    print('start time',start_time)
    for q_1,q_1_dot,q_1_dotdot,t  in zip(q_1s[1:],q_1_dots[1:],q_1_dotdots[1:],np.linspace(t0,Ts,num_steps)[1:]):
        msg_point = JointTrajectoryPoint()
        msg_point.positions = q_1
        msg_point.velocities = q_1_dot
        msg_point.accelerations = q_1_dotdot
        msg_point.time_from_start = ros.Duration.from_sec(t)
        goal_1.trajectory.points.append(msg_point)
        goal_1.goal_time_tolerance = tol
        # print(t0,Ts,num_steps)
    goal_2 = FollowJointTrajectoryGoal()
    goal_2.trajectory.joint_names = self.JOINT_NAMES
    for q_2,q_2_dot,q_2_dotdot,t  in zip(q_2s[1:],q_2_dots[1:],q_2_dotdots[1:],np.linspace(t0,Ts,num_steps)[1:]):
        msg_point = JointTrajectoryPoint()
        msg_point.positions = q_2
        msg_point.velocities = q_2_dot
        msg_point.accelerations = q_2_dotdot
        msg_point.time_from_start = ros.Duration.from_sec(t)
        goal_2.trajectory.points.append(msg_point)
        goal_2.goal_time_tolerance = tol
    goal_1.trajectory.header.stamp = ros.Time.from_sec(start_time)
    goal_2.trajectory.header.stamp = ros.Time.from_sec(start_time)
    # print(goal_1.trajectory)
    self.client_1.send_goal(goal_1)
    self.client_2.send_goal(goal_2)
planner.ros_handler.follow_pointwise_trajectory = partial(follow_pointwise_trajectory,planner.ros_handler)
def remove_shm_from_resource_tracker():
    """Monkey-patch multiprocessing.resource_tracker so SharedMemory won't be tracked

    More details at: https://bugs.python.org/issue38119
    """

    def fix_register(name, rtype):
        if rtype == "shared_memory":
            return
        return resource_tracker._resource_tracker.register(self, name, rtype)
    resource_tracker.register = fix_register

    def fix_unregister(name, rtype):
        if rtype == "shared_memory":
            return
        return resource_tracker._resource_tracker.unregister(self, name, rtype)
    resource_tracker.unregister = fix_unregister

    if "shared_memory" in resource_tracker._CLEANUP_FUNCS:
        del resource_tracker._CLEANUP_FUNCS["shared_memory"]
remove_shm_from_resource_tracker()
dtype = np.float32
num_tags_max = 10
array_shape = (4, 4, num_tags_max)
shared_memory_at_name = "apriltags_2"
try:
    shm_at = shared_memory.SharedMemory(name=shared_memory_at_name, create=True, size=np.prod(array_shape) * np.dtype(dtype).itemsize)
    shared_at = np.ndarray(array_shape, dtype=dtype, buffer=shm_at.buf)
    shared_at[:] = np.nan
    print("Created new shared memory.")
except FileExistsError:
    shm_at = shared_memory.SharedMemory(name=shared_memory_at_name)
    shared_at = np.ndarray(array_shape, dtype=dtype, buffer=shm_at.buf)
    print("Linked to existing shared memory.")

def setup_replan(self):
    
    self.replan_data.lam_g0[:] = self.initial_plan_data.lam_g
    self.replan_data.lam_x0[:] = self.initial_plan_data.lam_x
    self.replan_data.x0[:] = self.initial_plan_data.x
    bspline_1 = BSpline(
        self.initial_plan_data.result["robot_1_control_points"],
        self.order_mpc,
    )
    bspline_2 = BSpline(
        self.initial_plan_data.result["robot_2_control_points"],
        self.order_mpc,
    )
    bspline_obj = BSpline(
        self.initial_plan_data.result["carried_object_control_points"],
        self.order_mpc,
    )
    bspline_1_dot = bspline_1.fast_create_derivative_spline()
    bspline_2_dot = bspline_2.fast_create_derivative_spline()
    bspline_obj_dot = bspline_obj.fast_create_derivative_spline()
    # speed_scaling, actual_time = self.calculate_velocity_scaling(self.initial_plan_data.result['duration'].item(), self.max_speed, self.min_time, bspline_1_dot, bspline_2_dot)
    # self.replan_data.set_parameter('grasping_EE_1_bounds',np.repeat(self.initial_plan_data.result['grasping_x_EE_1'],2))
    # self.replan_data.set_parameter('grasping_EE_2_bounds',np.repeat(self.initial_plan_data.result['grasping_x_EE_2'],2))
    self.set_replan_grasping_position(self.initial_plan_data.result['grasping_x_EE_1'],self.initial_plan_data.result['grasping_x_EE_2'])
    self.implemented_trajectory['bspline_1'] = bspline_1
    self.implemented_trajectory['bspline_2'] = bspline_2
    self.implemented_trajectory['bspline_obj'] = bspline_obj
    self.implemented_trajectory['bspline_1_dot'] = bspline_1_dot
    self.implemented_trajectory['bspline_2_dot'] = bspline_2_dot
    self.implemented_trajectory['bspline_obj_dot'] = bspline_obj_dot
    self.implemented_trajectory['duration'] = self.initial_plan_data.result['duration'].item()
    self.implemented_trajectory['start_time'] = 0
    self.planned_trajectory['bspline_1'] = bspline_1
    self.planned_trajectory['bspline_2'] = bspline_2
    self.planned_trajectory['bspline_obj'] = bspline_obj
    self.planned_trajectory['bspline_1_dot'] = bspline_1_dot
    self.planned_trajectory['bspline_2_dot'] = bspline_2_dot
    self.planned_trajectory['bspline_obj_dot'] = bspline_obj_dot
    self.planned_trajectory['duration'] = self.initial_plan_data.result['duration'].item()
    self.planned_trajectory['start_time'] = 0

# planner.stop_recording_simulation = partial(stop_recording_simulation,planner)   


def replan(self,num_solves,t):
    t_start = time.perf_counter()
    
    next_t = t
    next_s = self.get_s(next_t)
    # next_s = 0.
    # print(next_s)
    q_1 = self.implemented_trajectory['bspline_1'].evaluate(next_s)
    q_2 = self.implemented_trajectory['bspline_2'].evaluate(next_s)
    q_obj = self.implemented_trajectory['bspline_obj'].evaluate(next_s)
    # q_1_dot = self.implemented_trajectory['bspline_1_dot'].evaluate(next_s)/self.implemented_trajectory['duration']
    # q_2_dot = self.implemented_trajectory['bspline_2_dot'].evaluate(next_s)/self.implemented_trajectory['duration']
    q_1_dot = self.implemented_trajectory['bspline_1_dot'].evaluate(next_s)/self.planned_trajectory['duration']
    q_2_dot = self.implemented_trajectory['bspline_2_dot'].evaluate(next_s)/self.planned_trajectory['duration']
    # self.set_placing_spot_visibility(self.lcm_subscription_handler.is_placing_spot_hidden or  (self._ignore_vision))
    self.set_start_position()
    # self.set_end_position()
    self.update_collision_svm(self.PlannerState.REPLAN)
    self.recompute_replan_bounds(q_1,q_2,q_obj,q_1_dot,q_2_dot)
    self.recompute_control_points_warm_start(next_t)
    self.replan_data.solve(
        warm=True, recompute_bounds=False
    )
    for z in range(num_solves - 1):
        self.replan_data.solve(
            warm=True, recompute_bounds=False
        )

    # self._new_plan_event.set()
    self.last_planning_time = time.perf_counter() - t_start

    bspline_1 = BSpline(
        self.replan_data.result["robot_1_control_points"],
        self.order_mpc,
    )
    bspline_2 = BSpline(
        self.replan_data.result["robot_2_control_points"],
        self.order_mpc,
    )
    bspline_obj = BSpline(
        self.replan_data.result["carried_object_control_points"],
        self.order_mpc,
    )
    # print('q_1_dot before replanning:',q_1_dot)
    # q_1_dot = bspline_1.fast_create_derivative_spline().evaluate(0)/self.replan_data.result['duration'].item()
    # print('q_1_dot after replanning:',q_1_dot)
    # print('q_1 after replanning:',bspline_1.evaluate(0))
    
    self.planned_trajectory['bspline_1'] = bspline_1
    self.planned_trajectory['bspline_2'] = bspline_2
    self.planned_trajectory['bspline_obj'] = bspline_obj        
    self.planned_trajectory['bspline_obj_dot'] = bspline_obj.fast_create_derivative_spline()   
    self.planned_trajectory['duration'] = self.replan_data.result['duration'].item()
    self.planned_trajectory['start_time'] = time.perf_counter()
    
planner.replan = partial(replan,planner)
# planner.set_robot_control_points_lbx_ubx = partial(set_robot_control_points_lbx_ubx,planner)
planner.setup_replan = partial(setup_replan,planner)
# planner.recompute_control_points_warm_start = partial(recompute_control_points_warm_start,planner)
planner.replan_data.bufferize_solver_and_bounds_g(solver = opt_collision_very_short.solver,lbg_ubg_function=opt_collision_very_short.lbg_ubg_func,lbx_ubx_function=opt_collision_very_short.lbx_ubx_func)
# planner.start_recording_simulation = partial(start_recording_simulation,planner)
# planner.stop_recording_simulation = partial(stop_recording_simulation,planner)
self = planner
self.set_simulation(True)
# planner._stop_event.set()
planner.set_time_bounds_replan(t_min = 0.01, t_max = 20.)




planner.num_steps_per_trajectory = 200
planner.last_planning_time = 0.3

planner.setup_replan()

bspline_1_initial = BSpline(
    self.initial_plan_data.result["robot_1_control_points"],
    self.order_mpc,
)
bspline_2_initial = BSpline(
    self.initial_plan_data.result["robot_2_control_points"],
    self.order_mpc,
)
bspline_obj_initial = BSpline(
    self.initial_plan_data.result["carried_object_control_points"],
    self.order_mpc,
)


robot_camera_depth_in_EE = np.array([[-0.00380074, -0.99995455, -0.00878683,  0.08132166],
       [ 0.99953527, -0.00406475,  0.03022601, -0.06282357],
       [-0.03026036, -0.00866787,  0.99950444, -0.073011  ],
       [ 0.        ,  0.        ,  0.        ,  1.        ]])
context = robots[0].plant.CreateDefaultContext()
EE_frame = robots[0].plant.GetFrameByName('EE_frame')
# robots[0].plant.SetPositions(context,)
def get_apriltag_robot_camera_frame_in_world(context,EE_frame, robot_camera_depth_in_EE,shared_at,robot_joints,):
    if robot_joints.size == 7:
        robot_joints = np.concatenate([robot_joints, np.array([0.,0])],)
    robots[0].plant.SetPositions(context,robot_joints)
    EE_in_world = EE_frame.CalcPoseInWorld(context).GetAsMatrix4()
    robot_depth_in_world = EE_in_world@robot_camera_depth_in_EE
    tags = shared_at.copy()
    tags_dict = {}
    # for _ in range(10):
    for i in range(shared_at.shape[2]):

        tag = tags[...,i]
        if np.isnan(tag[0,0]):
            break
        # print(tag)
        # T = tag[:3,:3]
        tag_T = tag.copy()
        tag_T[3,3]  = 1
        tag_T[3,0:3]  = 0
        # tag[3,3] = 1
        tag_T = robot_depth_in_world@tag_T
        R = tag_T[:3,:3]
        t = tag_T[:3,3]
        
        tag_id = round(tag[3,1])
        timestamp = tag[3,0]
        tags_dict[tag_id] = {'timestamp': timestamp,'R':R,'t':t}
    return tags_dict
get_apriltag_robot_camera_frame_in_world = partial(get_apriltag_robot_camera_frame_in_world,context,EE_frame, robot_camera_depth_in_EE,shared_at)
        # print(timestamp - time.perf_counter())
# get_apriltag_robot_camera_frame_in_world(planner.q_1s[-1][-1])

### This will make the robots move to the starting position using a simple joint space trajectory, beware of collisions, the total trajectory time is the last argument and can be made longer to play safer.

In [28]:
planner.ros_handler.minimum_time = 10
# planner.ros_handler.move_robots_to_configuration(bspline_1_initial.evaluate(0),bspline_2_initial.evaluate(0),10) #move both robots at the same time
# planner.ros_handler.move_robots_to_configuration(bspline_1_initial.evaluate(0),None,10) #move the first robot only
planner.ros_handler.move_robots_to_configuration(None,bspline_2_initial.evaluate(0),10) #move the second robot only

In [ ]:
# planner.lcm_subscription_handler.placing_spot_pose[:3,3:4]

### Run the next two cells in quick sequence, this will start the MPC.

In [ ]:

for opt_data in [planner.initial_plan_data, planner.parallel_solve_data, planner.replan_data]:
    weights = np.zeros(opt_data.parameter_shapes[name + '_weights'])
    support_vectors = np.zeros(opt_data.parameter_shapes[name + '_support_vectors'])
    weights[:, :polyharmonic_weights_np.shape[1]] = polyharmonic_weights_np.reshape(1,-1)
    support_vectors[:, :polyharmonic_sv_np.shape[1]] = polyharmonic_sv_np
    polynomial_weights_ = np.zeros(opt_data.parameter_shapes[name + '_polynomial_weights'])
    ww = polynomial_weights_np.reshape(-1,1)
    polynomial_weights_[:ww.shape[0]] = ww
    opt_data.set_parameter('vision_cost_weight',np.array(vision_weight))
    opt_data.set_parameter('robot_1_group_1_vision_polynomial_weights', polynomial_weights_)
    opt_data.set_parameter('robot_1_group_1_vision_weights', weights)
    opt_data.set_parameter('robot_1_group_1_vision_support_vectors', support_vectors)
    
    opt_data.set_parameter(
        "vision_cost_weight", np.array(vision_weight))
    opt_data.set_parameter(
        "end_position_lower_bounds",
        end_position_lower_bounds,
    )
    opt_data.set_parameter(
        "end_position_upper_bounds",
        end_position_upper_bounds,
    )
    opt_data.set_parameter(
    "end_angle_bounds",
    end_angle_bounds,
)

meshcat.DeleteRecording()
meshcat.Delete(
        f"/last_plan_warm/"
    )
for i in range(100):
    meshcat.Delete(f'/last_point/{i}')
    meshcat.Delete(f'/first_point/{i}')
    meshcat.Delete(f'/obj_trajectory/{i}')
planner.setup_replan()
# planner.start_replaning()
point_cloud_viz.update()
planner.replan_data.set_parameter("object_end_position", planner.lcm_subscription_handler.placing_spot_pose[:3,3:4])
planner.replan_data.set_parameter("object_end_rotation_matrix", planner.lcm_subscription_handler.placing_spot_pose[:3,:3])

planner.set_costs(vision_cost = -1000, duration_cost = 0.1, acceleration_cost = 0.01, manipulability_cost = .0001,replan_connection_cost = 3000.,slack_cost_weight = -100.)
t_start = time.perf_counter()
Ts = 0.4
stuff = []
t_sim = 0
planner.start_recording_simulation()
# self.set_max_speed(5)
planner.set_velocity_scaling(0.2)
last_t = time.perf_counter()
Tss = []

planner.set_collision_bounds(-np.inf,.3)
planner.set_simulation(False)
# adfsdfas
# point_cloud_viz.update()
planner.obstacles.append(point_cloud_viz.point_cloud.copy())
# planner.replan_data.set_initial_guess('robot_1_group_1_svm_slack',np.array(0.))
# planner.replan_data.set_initial_guess('robot_1_group_2_svm_slack',np.array(0.))
# planner.replan_data.set_initial_guess('robot_1_group_3_svm_slack',np.array(0.))
# planner.replan_data.set_initial_guess('robot_1_group_4_svm_slack',np.array(0.))
# planner.replan_data.set_initial_guess('robot_2_group_1_svm_slack',np.array(0.))
# planner.replan_data.set_initial_guess('robot_2_group_2_svm_slack',np.array(0.))
# planner.replan_data.set_initial_guess('robot_2_group_3_svm_slack',np.array(0.))
# planner.replan_data.set_initial_guess('robot_2_group_4_svm_slack',np.array(0.))
planner.replan(10,0)
planned_bspline_1 = self.planned_trajectory['bspline_1']
planned_bspline_2 = self.planned_trajectory['bspline_2']

bspline_1, bspline_2 = planned_bspline_1,planned_bspline_2
bspline_1_dot = bspline_1.fast_create_derivative_spline()
bspline_2_dot = bspline_2.fast_create_derivative_spline()
# speed_scaling, actual_time = self.calculate_velocity_scaling(planned_trajectory_duration, self.max_speed, self.min_time, bspline_1_dot, bspline_2_dot)
self.implemented_trajectory['bspline_1'] = bspline_1
self.implemented_trajectory['bspline_2'] = bspline_2
self.implemented_trajectory['bspline_obj'] = self.planned_trajectory['bspline_obj']
self.implemented_trajectory['bspline_1_dot'] = bspline_1_dot
self.implemented_trajectory['bspline_2_dot'] = bspline_2_dot
self.implemented_trajectory['duration'] = self.planned_trajectory['duration']
start_time = ros.Time.now().to_time() + 1
self.implemented_trajectory['start_time'] = start_time
s = 0
s_next = self.get_s(start_time + Ts)
q_objs_all = []
s_ = np.linspace(s,s_next,self.num_steps_per_trajectory)
q_1s = self.implemented_trajectory['bspline_1'].fast_batch_evaluate(s_)
q_2s = self.implemented_trajectory['bspline_2'].fast_batch_evaluate(s_)
times = []
actual_q1s = []
actual_q2s = []
actual_q1_dots = []
actual_q2_dots = []
if self.simulation:
    ts = np.linspace(self.implemented_trajectory['start_time'] - start_time,self.implemented_trajectory['start_time'] - start_time+Ts,q_1s.shape[0]) 
    self.q_1s.append(q_1s)
    self.q_2s.append(q_2s)
    self.q_obj.append(self.implemented_trajectory['bspline_obj'].fast_batch_evaluate(s_))
    self.ts.append(ts)
    
    q_objs_all.append(self.planned_trajectory['bspline_obj'].control_points.copy())
    
else:
    q_1_dots = self.implemented_trajectory['bspline_1_dot'].fast_batch_evaluate(s_)/self.implemented_trajectory['duration']
    q_2_dots = self.implemented_trajectory['bspline_2_dot'].fast_batch_evaluate(s_)/self.implemented_trajectory['duration']
    q_1_dotdots = self.implemented_trajectory['bspline_1_dot'].fast_create_derivative_spline().fast_batch_evaluate(s_)/self.implemented_trajectory['duration']**2
    q_2_dotdots = self.implemented_trajectory['bspline_2_dot'].fast_create_derivative_spline().fast_batch_evaluate(s_)/self.implemented_trajectory['duration']**2
    print('time_now',ros.Time.now().to_time())
    self.ros_handler.follow_pointwise_trajectory(q_1s,q_2s,q_1_dots,q_2_dots,q_1_dotdots,q_2_dotdots,Ts,self.implemented_trajectory['start_time'])
    ts = np.linspace(self.implemented_trajectory['start_time'] - start_time,self.implemented_trajectory['start_time'] - start_time +Ts,q_1s.shape[0]) 
    self.q_1s.append(q_1s)
    self.q_2s.append(q_2s)
    self.q_obj.append(self.implemented_trajectory['bspline_obj'].fast_batch_evaluate(s_))
    self.ts.append(ts)
    
    q_objs_all.append(self.planned_trajectory['bspline_obj'].control_points.copy())


    indices = round(Ts/(1/planner.ros_handler.topic_frequency))
    # actual_q1_dots.append(np.array(planner.ros_handler.joints_dot_1_filtered_history)[-indices:])
    # actual_q2_dots.append(    np.array(planner.ros_handler.joints_dot_2_filtered_history)[-indices:])
    actual_q1s.append(    np.array(planner.ros_handler.joints_1_history)[-indices:])
    actual_q2s.append(    np.array(planner.ros_handler.joints_2_history)[-indices:])
publish_trajectory(planner.replan_data, planner.lcm_subscription_handler, order = 3)
# publish_trajectory(planner.replan_data, planner.lcm_subscription_handler, order = 3)
# print(self.replan_data.print_violated_g())

# bspline_1_col = BSpline(
#     planner.replan_data.result["robot_1_control_points"],
#     opt_collision.order,
# )
# bspline_2_col = BSpline(
#     planner.replan_data.result["robot_2_control_points"],
#     opt_collision.order,
# )
# bspline_obj_col = BSpline(
#     planner.replan_data.result["carried_object_control_points"],
#     opt_collision.order,
# )

# meshcat.StartRecording(frames_per_second=30.0)

# for s in np.linspace(0,1,15):
#     q_1 = bspline_1_col.evaluate(s)
#     q_1 = np.concatenate([q_1,np.zeros(2)])
#     q_2 = bspline_2_col.evaluate(s)
#     q_2 = np.concatenate([q_2,np.zeros(2)])
#     q_obj = bspline_obj_col.evaluate(s)
#     viz_helper.set_position('robot_0',q_1)
#     viz_helper.set_position('robot_1',q_2)
#     viz_helper.set_position('carried_object',q_obj)
    

#     viz_helper.diagram_context.SetTime(s)
#     viz_helper.publish_diagram()

# meshcat.StopRecording()
# meshcat.PublishRecording()

# i = 0
# print(BSpline(
#     planner.replan_data.result["robot_1_control_points"],
#     opt_collision.order,
# ).evaluate(0) - BSpline(
#     planner.initial_plan_data.result["robot_1_control_points"],
#     opt_collision.order,
# ).evaluate(0))
# time start loop 1723905033.311365
# time_now 1723905033.5862064
# start time 1723905033.3746598

In [ ]:
# tags[tag_index]['t'] + tag_offset

In [ ]:
# %%prun
got_tag = False
tag_index = 0
tag_offset = np.array([0.12,0.00,0.15])
got_tag_at_t = 0
with threadpoolctl.threadpool_limits(limits=12):
    for i in range(300):
        if i >= 10 and not got_tag:
            tags = get_apriltag_robot_camera_frame_in_world(planner.ros_handler.joints_1)
            # tags = get_apriltag_robot_camera_frame_in_world(planner.q_1s[-1][-1])
            planner.set_velocity_scaling(0.1)
            if tag_index in tags and time.perf_counter() - tags[tag_index]['timestamp'] < 2.:
                print('GOT TAGGGGG')
                print('GOT TAGGGGG')
                print('GOT TAGGGGG')
                got_tag = True
                got_tag_at_t = ros.Time.now().to_time() - start_time
        # if False:
                # planner.lcm_subscription_handler.placing_spot_pose = tags[0]
                planner.replan_data.set_parameter("object_end_position", tags[tag_index]['t'] + tag_offset)
                # planner.replan_data.set_parameter("object_end_rotation_matrix", tags[tag_index]['R'])
                for opt_data in [planner.initial_plan_data, planner.parallel_solve_data, planner.replan_data]:
                    weights = np.zeros(opt_data.parameter_shapes[name + '_weights'])
                    support_vectors = np.zeros(opt_data.parameter_shapes[name + '_support_vectors'])
                    weights[:, :polyharmonic_weights_np.shape[1]] = polyharmonic_weights_np.reshape(1,-1)
                    support_vectors[:, :polyharmonic_sv_np.shape[1]] = polyharmonic_sv_np
                    polynomial_weights_ = np.zeros(opt_data.parameter_shapes[name + '_polynomial_weights'])
                    ww = polynomial_weights_np.reshape(-1,1)
                    polynomial_weights_[:ww.shape[0]] = ww
                    opt_data.set_parameter('vision_cost_weight',np.array(vision_weight))
                    opt_data.set_parameter('robot_1_group_1_vision_polynomial_weights', polynomial_weights_)
                    opt_data.set_parameter('robot_1_group_1_vision_weights', weights)
                    opt_data.set_parameter('robot_1_group_1_vision_support_vectors', support_vectors)
                    
                    opt_data.set_parameter(
                        "vision_cost_weight", np.array(vision_weight)*0)
                    opt_data.set_parameter(
                        "end_position_lower_bounds",
                        np.array([-0.02,-0.02,-0.06]),
                    )
                    opt_data.set_parameter(
                        "end_position_upper_bounds",
                        np.array([0.02,0.02,0.06]),
                    )
                    opt_data.set_parameter(
                    "end_angle_bounds",
                    np.array(0.1),
                )
        t_start_loop = time.perf_counter()
        print('time start loop',ros.Time.now().to_time())
        planner.obstacles.append(point_cloud_viz.point_cloud.copy())
        # planner.replan_data.set_initial_guess('robot_1_group_1_svm_slack',np.array(0.))
        # planner.replan_data.set_initial_guess('robot_1_group_2_svm_slack',np.array(0.))
        # planner.replan_data.set_initial_guess('robot_1_group_3_svm_slack',np.array(0.))
        # planner.replan_data.set_initial_guess('robot_1_group_4_svm_slack',np.array(0.))
        # planner.replan_data.set_initial_guess('robot_2_group_1_svm_slack',np.array(0.))
        # planner.replan_data.set_initial_guess('robot_2_group_2_svm_slack',np.array(0.))
        # planner.replan_data.set_initial_guess('robot_2_group_3_svm_slack',np.array(0.))
        # planner.replan_data.set_initial_guess('robot_2_group_4_svm_slack',np.array(0.))
        planner.replan(1,self.implemented_trajectory['start_time'] + Ts)
        print(planner.replan_data.within_bounds())
        # print('t1',time.perf_counter()-t_start_loop)

        planned_bspline_1 = self.planned_trajectory['bspline_1']
        planned_bspline_2 = self.planned_trajectory['bspline_2']
        planned_trajectory_duration = self.planned_trajectory['duration']
        bspline_1, bspline_2 = planned_bspline_1,planned_bspline_2
        bspline_1_dot = bspline_1.fast_create_derivative_spline()
        bspline_2_dot = bspline_2.fast_create_derivative_spline()
        # speed_scaling, actual_time = self.calculate_velocity_scaling(planned_trajectory_duration, self.max_speed, self.min_time, bspline_1_dot, bspline_2_dot)
        self.implemented_trajectory['bspline_1'] = bspline_1
        self.implemented_trajectory['bspline_2'] = bspline_2
        self.implemented_trajectory['bspline_obj'] = self.planned_trajectory['bspline_obj']
        self.implemented_trajectory['bspline_1_dot'] = bspline_1_dot
        self.implemented_trajectory['bspline_2_dot'] = bspline_2_dot
        self.implemented_trajectory['duration'] = self.planned_trajectory['duration']
        # self.implemented_trajectory['start_time'] = time.perf_counter()
        self.implemented_trajectory['start_time'] += Ts
        # t_now = time.perf_counter()
        # print('tnow',t_now)

        # print('t2',time.perf_counter()-t_start_loop)
        s = 0.
        s_next = self.get_s(self.implemented_trajectory['start_time'] + Ts)
        s_next = min(s_next, 1)
        s_ = np.linspace(s,s_next,self.num_steps_per_trajectory)
        q_1s = self.implemented_trajectory['bspline_1'].fast_batch_evaluate(s_)
        q_2s = self.implemented_trajectory['bspline_2'].fast_batch_evaluate(s_)
        
        # self.check_path_for_collision()
        if self.simulation:
            ts = np.linspace(self.implemented_trajectory['start_time']- start_time,self.implemented_trajectory['start_time']+Ts - start_time,q_1s.shape[0]) 
            self.q_1s.append(q_1s[5:])
            self.q_2s.append(q_2s[5:])
            q_obj = self.implemented_trajectory['bspline_obj'].fast_batch_evaluate(s_)
            # q_obj[0:1,-1] += 1.
            self.q_obj.append(q_obj[5:])
            self.ts.append(ts[5:])
            print(ts[0]-self.ts[0][0],ts[-1]-self.ts[0][0])
            q_objs_all.append(self.planned_trajectory['bspline_obj'].control_points.copy())
            # print('t3',time.perf_counter()-t_start_loop)
            # times.append(actual_time)
            # print(self.implemented_trajectory['start_time'] - last_t)
        else:
            q_1_dots = self.implemented_trajectory['bspline_1_dot'].fast_batch_evaluate(s_)/self.implemented_trajectory['duration']
            q_2_dots = self.implemented_trajectory['bspline_2_dot'].fast_batch_evaluate(s_)/self.implemented_trajectory['duration']
            # self.ros_handler.follow_pointwise_trajectory(q_1s,q_2s,q_1_dots,q_2_dots,Ts)+
            q_1_dotdots = self.implemented_trajectory['bspline_1_dot'].fast_create_derivative_spline().fast_batch_evaluate(s_)/self.implemented_trajectory['duration']**2
            q_2_dotdots = self.implemented_trajectory['bspline_2_dot'].fast_create_derivative_spline().fast_batch_evaluate(s_)/self.implemented_trajectory['duration']**2
            print('time_now',ros.Time.now().to_time())
            self.ros_handler.follow_pointwise_trajectory(q_1s,q_2s,q_1_dots,q_2_dots,q_1_dotdots,q_2_dotdots,Ts, self.implemented_trajectory['start_time'])
            ts = np.linspace(self.implemented_trajectory['start_time'] - start_time,self.implemented_trajectory['start_time'] - start_time +Ts,q_1s.shape[0]) 
            self.q_1s.append(q_1s)
            self.q_2s.append(q_2s)
            q_obj = self.implemented_trajectory['bspline_obj'].fast_batch_evaluate(s_)
            self.q_obj.append(q_obj)
            self.ts.append(ts)
            q_objs_all.append(self.planned_trajectory['bspline_obj'].control_points.copy())
            # indices = round(Ts/(1/planner.ros_handler.topic_frequency))
            
        publish_trajectory(planner.replan_data, planner.lcm_subscription_handler, order = 3)
        actual_Ts = time.perf_counter() - t_start_loop
        # print(actual_Ts)
        Tss.append(actual_Ts)
        # if not self.simulation:
        time.sleep(max(0.00001,Ts-actual_Ts))
        indices = round(Ts/(1/planner.ros_handler.topic_frequency))
        # actual_q1_dots.append(np.array(planner.ros_handler.joints_dot_1_filtered_history)[-indices:])
        # actual_q2_dots.append(    np.array(planner.ros_handler.joints_dot_2_filtered_history)[-indices:])
        actual_q1s.append(    np.array(planner.ros_handler.joints_1_history)[-indices:])
        actual_q2s.append(    np.array(planner.ros_handler.joints_2_history)[-indices:])
        # point_cloud_viz.update()
        print(self.replan_data.print_violated_g())
        # print(self.replan_data.result['robot_1_group_4_svm_slack'])
        # print(self.replan_data.result['robot_2_group_4_svm_slack'])
        
# i += 1
# bspline_1_col = BSpline(
#     planner.replan_data.result["robot_1_control_points"],
#     opt_collision.order,
# )
# bspline_2_col = BSpline(
#     planner.replan_data.result["robot_2_control_points"],
#     opt_collision.order,
# )
# bspline_obj_col = BSpline(
#     planner.replan_data.result["carried_object_control_points"],
#     opt_collision.order,
# )
# planner.stop_recording_simulation()


# meshcat.StartRecording(frames_per_second=30.0)

# for s in np.linspace(0,1,15):
#     q_1 = bspline_1_col.evaluate(s)
#     q_1 = np.concatenate([q_1,np.zeros(2)])
#     q_2 = bspline_2_col.evaluate(s)
#     q_2 = np.concatenate([q_2,np.zeros(2)])
#     q_obj = bspline_obj_col.evaluate(s)
#     viz_helper.set_position('robot_0',q_1)
#     viz_helper.set_position('robot_1',q_2)
#     viz_helper.set_position('carried_object',q_obj)
    

#     viz_helper.diagram_context.SetTime(s)
#     viz_helper.publish_diagram()
# q_obj = bspline_obj_col.fast_batch_evaluate(np.linspace(0,1,40))
# q_first = q_obj[0]
# q_last = bspline_obj_col.evaluate(s_next)
# meshcat.SetObject(
#     f'/first_point/{i}', Sphere(0.01)
# )
# meshcat.SetProperty(
#     f'/first_point/{i}', 'position', q_first[4:]
# )
# # meshcat.SetProperty(f"thing/{i}", "color",[1,0,0,1])
# meshcat.SetProperty(f"/first_point/{i}", "color", [1, 0, 0, 0.5])
# # print( q_first[3:])
# meshcat.SetObject(
#     f'/last_point/{i}', Sphere(0.01)
# )
# meshcat.SetProperty(
#     f'/last_point/{i}', 'p.0, 0.27, 0.25],rpy=RollPitchYaw([0, 0, np.pi/4*0])) # behind camera
# end_pose = RigidTransform(p=[0.0obj_trajectory/'+str(i),q_obj[:-1,4:].T,q_obj[1:,4:].T,)
# meshcat.StopRecording()
# meshcat.PublishRecording()

In [ ]:
Ts

In [194]:
# planner.ros_handler.move_robots_to_configuration(planner.q_1s[0][0],None,10)
# planner.ros_handler.move_robots_to_configuration(None,planner.q_2s[0][0],10)

### plot in meshcat the obj trajectory

In [47]:
for i,(q_obj_cp) in enumerate(q_objs_all):
    b_spline = BSpline(q_obj_cp,3)
    q_obj = b_spline.fast_batch_evaluate(np.linspace(0,1,40))
    # meshcat.SetLineSegments('x1',T@vector_origin,T@vector_x,rgba=Rgba(1,0,0,1))
    meshcat.SetLineSegments('/obj_trajectory/'+str(i),q_obj[:-1,4:].T,q_obj[1:,4:].T,)

In [ ]:
# bspline_1_col = BSpline(
#     planner.replan_data.result["robot_1_control_points"],
#     opt_collision.order,
# )
# bspline_2_col = BSpline(
#     planner.replan_data.result["robot_2_control_points"],
#     opt_collision.order,
# )
# bspline_obj_col = BSpline(
#     planner.replan_data.result["carried_object_control_points"],
#     opt_collision.order,
# )
# print_collision_score_on_samples(20,planner,bspline_1_col,bspline_2_col)

### Show the recorded motion in meshcat, might take a bit

In [ ]:

for i,(q_obj_cp) in enumerate(q_objs_all):
    b_spline = BSpline(q_obj_cp,3)
    q_obj = b_spline.fast_batch_evaluate(np.linspace(0,1,40))
    # meshcat.SetLineSegments('x1',T@vector_origin,T@vector_x,rgba=Rgba(1,0,0,1))
    meshcat.SetLineSegments('/obj_trajectory/'+str(i),q_obj[:-1,4:].T,q_obj[1:,4:].T,)
planner.stop_recording_simulation()

In [ ]:
len(planner.ros_handler.joints_1_history)

In [41]:
# from collections import deque
# # planner.ros_handler.joints_1_history
# max_store_seconds = 100
# max_size = max_store_seconds*planner.ros_handler.topic_frequency
# planner.ros_handler.joints_1_history = deque(maxlen=max_size)  
# planner.ros_handler.joints_2_history = deque(maxlen=max_size)  
# planner.ros_handler.joints_dot_1_history = deque(maxlen=max_size)  
# planner.ros_handler.joints_dot_2_history = deque(maxlen=max_size)  
# planner.ros_handler.joints_dot_1_filtered_history = deque(maxlen=max_size)  
# planner.ros_handler.joints_dot_2_filtered_history = deque(maxlen=max_size)  

In [75]:
# planner.ros_handler.move_robots_to_configuration(planner.q_1s[0][0],planner.q_2s[0][0],10)



In [ ]:
plt.plot((Tss),'--')
got_tag
# point_cloud_viz.update()

In [ ]:
planner.q_1s

In [ ]:
s = np.vstack(actual_q1s)
plt.plot(np.linspace(planner.ts[0][0]-2.,planner.ts[-1][-1],s.shape[0])*0.95,s)
plt.plot(np.hstack(planner.ts),np.vstack(planner.q_1s), '--')

In [ ]:
robot_1_collision_checker = torch.compile(robots[0].collision_model.link_collision_modules["all_geometries"].to('cpu'),dynamic=True)
robot_2_collision_checker = torch.compile(robots[1].collision_model.link_collision_modules["all_geometries"].to('cpu'),dynamic=True)
# cc = 
num_eval_points = 20
q_1 = torch.as_tensor(np.column_stack((bspline_1.fast_batch_evaluate(np.linspace(0,1,num_eval_points)), np.zeros((num_eval_points,2)))))
q_2 = torch.as_tensor(np.column_stack((bspline_2.fast_batch_evaluate(np.linspace(0,1,num_eval_points)), np.zeros((num_eval_points,2)))))

obstacle_points = torch.randn(400,3)
obstacle_radii = torch.tensor(0.04, dtype=torch.float32, device='cpu').expand(400, 1)

robot_1_collision_checker(q_1,obstacle_points,obstacle_radii)
robot_2_collision_checker(q_2,obstacle_points,obstacle_radii)

In [92]:
meshcat.SetProperty("/drake/contact_forces",'visible',False)
meshcat.SetProperty("/drake/proximity",'visible',False)

In [ ]:
print_collision_score_on_samples(10,planner,bspline_1_col,bspline_2_col)
# planner.ros_handler.topic_frequency = 30

In [19]:
def follow_pointwise_trajectory(self,q_1s,q_2s,q_1_dots,q_2_dots,q_1_dotdots,q_2_dotdots, Ts, start_time):
    # if self.simulation:
    #     for q_1,q_1_dot,q_2,q_2_dot,t  in zip(q_1s,q_2_dots,q_1s,q_2_dots,np.linspace(0,time,num_steps)):
    #         self.viz_helper.set_position('robot_1',q_1)
    #         self.viz_helper.set_position('robot_2',q_2)
    #         self.viz_helper.diagram_context.SetTime(self.simulation_time + t)
    #         self.viz_helper.publish()
    #     self.simulation_time += time
    #     return 
    # http://wiki.ros.org/joint_trajectory_controller/UnderstandingTrajectoryReplacement
    goal_1 = FollowJointTrajectoryGoal()
    goal_1.trajectory.joint_names = self.JOINT_NAMES
    num_steps = len(q_1s)
    tol = ros.Duration.from_sec(0.01)
    t0 = 0.00
    Ts += t0
    print('start time',start_time)
    for q_1,q_1_dot,q_1_dotdot,t  in zip(q_1s[1:],q_1_dots[1:],q_1_dotdots[1:],np.linspace(t0,Ts,num_steps)[1:]):
        msg_point = JointTrajectoryPoint()
        msg_point.positions = q_1
        msg_point.velocities = q_1_dot
        msg_point.accelerations = q_1_dotdot
        msg_point.time_from_start = ros.Duration.from_sec(t)
        goal_1.trajectory.points.append(msg_point)
        goal_1.goal_time_tolerance = tol
        # print(t0,Ts,num_steps)
    goal_2 = FollowJointTrajectoryGoal()
    goal_2.trajectory.joint_names = self.JOINT_NAMES
    for q_2,q_2_dot,q_2_dotdot,t  in zip(q_2s[1:],q_2_dots[1:],q_2_dotdots[1:],np.linspace(t0,Ts,num_steps)[1:]):
        msg_point = JointTrajectoryPoint()
        msg_point.positions = q_2
        msg_point.velocities = q_2_dot
        msg_point.accelerations = q_2_dotdot
        msg_point.time_from_start = ros.Duration.from_sec(t)
        goal_2.trajectory.points.append(msg_point)
        goal_2.goal_time_tolerance = tol
    goal_1.trajectory.header.stamp = ros.Time.from_sec(start_time)
    goal_2.trajectory.header.stamp = ros.Time.from_sec(start_time)
    # print(goal_1.trajectory)
    self.client_1.send_goal(goal_1)
    self.client_2.send_goal(goal_2)
planner.ros_handler.follow_pointwise_trajectory = partial(follow_pointwise_trajectory,planner.ros_handler)

In [50]:
import pickle
# with open('ffff','wb') as f:
#     pickle.dump((got_tag_at_t,self.obstacles,self.q_1s,self.q_2s,self.ts,actual_q1s,actual_q2s,Tss),f)
with open('vision_scenario_redone','wb') as f:
    pickle.dump((got_tag_at_t,self.obstacles,self.q_1s,self.q_2s,self.ts,actual_q1s,actual_q2s,Tss,q_objs_all),f)
# with open('ffff','rb') as handle:
#     b = pickle.load(handle)
# b

In [20]:
def stop_recording_simulation(self):
    q_1s = np.concatenate(self.q_1s,axis= 0 )
    q_2s = np.concatenate(self.q_2s,axis= 0 )
    q_objs = np.concatenate(self.q_obj,axis= 0 )
    ts = np.concatenate(self.ts,axis= 0 )
    dt = self.ts[0][-1] - self.ts[0][0]
    self.meshcat.StartRecording(frames_per_second=30.0)
    k = 0
    self.meshcat.Delete('/obstacles')
    obstacle_radii = 0.04
    # self.meshcat.Delete(f"/obstacles/{i}")
    # for i in range(500):
    for i in range(1000):
        # for i, q_sample in enumerate(point_clo ud):
        self.meshcat.SetObject(
            f"/obstacles/{i}", Sphere(obstacle_radii)
        )
        self.meshcat.SetProperty(
            f"/obstacles/{i}", "position", [0,0,0], time_in_recording = 0,
        )
    for i in range(q_1s.shape[0]):
        t = ts[i] - ts[0]
        if t - k*dt > 0:
            k += 1
            try:
                obstacles = self.obstacles[k]
                obstacles = obstacles[~np.isnan(obstacles).any(axis=1)]
                # print(obstacles.shape)
                for j in range(obstacles.shape[0]):
                    self.meshcat.SetProperty(
                            f"/obstacles/{j}", "position", obstacles[j], time_in_recording = t - ts[0],
                        )
                for j in range(obstacles.shape[0],1000):
                    self.meshcat.SetProperty(
                            f"/obstacles/{j}", "position", [0,0,0], time_in_recording = t - ts[0],
                        )
            except:
                print(k)
        q_1 = q_1s[i]
        q_1 = np.concatenate([q_1,np.zeros(2)])

        q_2 = q_2s[i]
        q_2 = np.concatenate([q_2,np.zeros(2)])
        q_obj = q_objs[i]
        self.viz_helper.set_position('robot_0',q_1)
        self.viz_helper.set_position('robot_1',q_2)
        self.viz_helper.set_position('carried_object',q_obj)
        self.viz_helper.diagram_context.SetTime(ts[i] - ts[0])
        self.viz_helper.publish()
    for i,q_obj in enumerate(self.q_obj):
        q_first = q_obj[0]
        q_last = q_obj[-1]
        self.meshcat.SetObject(
            f'/first_point/{i}', Sphere(0.01)
        )
        self.meshcat.SetProperty(
            f'/first_point/{i}', 'position', q_first[4:]
        )
        # self.meshcat.SetProperty(f"thing/{i}", "color",[1,0,0,1])
        self.meshcat.SetProperty(f"/first_point/{i}", "color", [1, 0, 0, 0.5])
        # print( q_first[3:])
        self.meshcat.SetObject(
            f'/last_point/{i}', Sphere(0.01)
        )
        self.meshcat.SetProperty(
            f'/last_point/{i}', 'position', q_last[4:]
        )
        self.meshcat.SetProperty(f"/last_point/{i}", "color", [0,1, 0, 0.5])
    self.meshcat.StopRecording()
    self.meshcat.PublishRecording()
def start_recording_simulation(self):
    self.q_1s = []
    self.q_2s = []
    self.q_obj = []
    self.obstacles = []
    self.ts = []
    self.meshcat.StartRecording(frames_per_second=30.0)
planner.start_recording_simulation = partial(start_recording_simulation,planner)
planner.stop_recording_simulation = partial(stop_recording_simulation,planner)
# meshcat.SetPro
# point_cloud_viz.point_cloud

In [ ]:
plt.plot(Tss)
# planner.stop_recording_simulation()

In [ ]:
meshcat.web_url()

In [ ]:

s_ = np.linspace(0,1,15)
q_1s = self.implemented_trajectory['bspline_1'].fast_batch_evaluate(s_)
q_2s = self.implemented_trajectory['bspline_2'].fast_batch_evaluate(s_)
q_objs = self.implemented_trajectory['bspline_obj'].fast_batch_evaluate(s_)
# self.check_path_for_collision()

meshcat.StartRecording(frames_per_second=30.0)

for t,q_1,q_2,q_obj in zip(np.linspace(0,1,15),q_1s,q_2s,q_objs):
    
    q_1 = np.concatenate([q_1,np.zeros(2)])
    
    q_2 = np.concatenate([q_2,np.zeros(2)])
    # q_obj = bspline_obj_col.evaluate(s)
    
    # print(g_score_fake_function_repsum.call({'x':np.concatenate([q_obj,q_1,q_2,np.zeros(6)]),'fk_S':opt_data_collision.parameters[opt_info.svm_support_vectors],'A':opt_data_collision.parameters[opt_info.svm_weights]}))
    # print(g_translation)
    viz_helper.set_position('robot_0',q_1)
    viz_helper.set_position('robot_1',q_2)
    viz_helper.set_position('carried_object',q_obj)
    

    viz_helper.diagram_context.SetTime(t)
    # viz_helper.publish()
    viz_helper.publish_diagram()
meshcat.StopRecording()
# print('Success?',opt_collision.optimization_data['debug'].within_bounds())

# print(opt_collision.optimization_data['debug'].result["duration"])
meshcat.PublishRecording()

In [ ]:
planner.stop_recording_simulation()